In [1]:
import pandas as pd
import numpy as np
import sys
import os
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None) 
#pd.set_option('display.max_info_columns', None)

sys.path.append(os.path.abspath(".."))

# Import the function
from scripts.utils import (get_variable_name,
                           count_repetitive_caseids,
                            add_prefix_except_caseid, 
                            get_dummy_variables, 
                            convert_objects_to_int64_safe,
                            drop_null_and_list,
                            aggregate_sum_by_caseid,
                            analyze_matches,
                            analyze_matches_explicit_keys, 
                            aggregate_with_value_suffix          
                            ) 


# 01 Target

In [2]:
Modulo1632_REC21_2023 = pd.read_csv("C:/Users/linoc/OneDrive/Encoder/03_partos/01_raws/2024/968-Modulo1632/968-Modulo1632/REC21_2024.csv", low_memory=False)
Modulo1632_REC21_2023.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60693 entries, 0 to 60692
Data columns (total 34 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   ID1      60693 non-null  int64 
 1   CASEID   60693 non-null  object
 2   BIDX     60693 non-null  int64 
 3   BORD     60693 non-null  int64 
 4   B0       60693 non-null  int64 
 5   B1       60693 non-null  int64 
 6   B2       60693 non-null  int64 
 7   B3       60693 non-null  object
 8   B4       60693 non-null  int64 
 9   B5       60693 non-null  int64 
 10  B6       60693 non-null  object
 11  B7       60693 non-null  object
 12  B8       60693 non-null  object
 13  B9       60693 non-null  object
 14  B10      60693 non-null  int64 
 15  B11      60693 non-null  object
 16  B12      60693 non-null  object
 17  B13      60693 non-null  object
 18  B15      60693 non-null  object
 19  B16      60693 non-null  object
 20  BD       60693 non-null  int64 
 21  BDD      60693 non-null  int64 
 22

In [3]:
target = Modulo1632_REC21_2023[['CASEID', 'BIDX','Q220A']]
target.CASEID.nunique(), target[target['Q220A']!=" "].CASEID.nunique()

(24627, 18335)

In [4]:
# # de meses de nascimento
target.Q220A.value_counts()

Q220A
     39575
9    16932
8     3467
7      580
6      122
5       17
Name: count, dtype: int64

In [5]:
target=target[target['Q220A']!= " " ]
target['Q220A'] = pd.to_numeric(target['Q220A'])
target.Q220A.value_counts()

Q220A
9    16932
8     3467
7      580
6      122
5       17
Name: count, dtype: int64

In [6]:
target.CASEID.nunique(), target.shape

(18335, (21118, 3))

In [7]:
# Keep row with max BIDX per CASEID
target = target.loc[target.groupby("CASEID")["BIDX"].idxmax()]

# Reset index (optional, just for clean display)
target = target.reset_index(drop=True)

#Create flag 
target['premature'] =  np.where(target['Q220A'] <= 8, 1, 0)
target_final = target.copy()

#target_final = target.groupby('CASEID', as_index=False)['premature'].sum()
target_final['premature_flag'] = np.where(target_final['premature'] >= 1, 1, 0)
target_final.premature_flag.value_counts()

premature_flag
0    14801
1     3534
Name: count, dtype: int64

In [8]:
target_final.premature_flag.value_counts(1)

premature_flag
0    0.807254
1    0.192746
Name: proportion, dtype: float64

In [9]:
target_final.shape , target_final.CASEID.nunique()

((18335, 5), 18335)

In [10]:
output_file = "target_final.csv"

# Go one level up from the current working directory
base_dir = os.path.dirname(os.getcwd())   # gives "c:\\Users\\linoc\\OneDrive\\Encoder\\03_partos"
output_dir = os.path.join(base_dir, "data\\interim")
print(output_dir)

#Full path
output_path = os.path.join(output_dir, output_file)

# Save DataFrame
target_final.to_csv(output_path, index=False, encoding="utf-8-sig")

c:\Users\linoc\OneDrive\Encoder\03_partos\02_scripts\Premature_model\data\interim


# 2 join other tables

In [11]:
Modulo1631_REC91_2024 = pd.read_csv("C:/Users/linoc/OneDrive/Encoder/03_partos/01_raws/2024/968-Modulo1631/968-Modulo1631/REC91_2024.csv", low_memory=False)
Modulo1632_RE223132_2024 = pd.read_csv("C:/Users/linoc/OneDrive/Encoder/03_partos/01_raws/2024/968-Modulo1632/968-Modulo1632/RE223132_2024.csv", low_memory=False)
Modulo1633_REC41_2024 = pd.read_csv("C:/Users/linoc/OneDrive/Encoder/03_partos/01_raws/2024/968-Modulo1633/968-Modulo1633/REC41_2024.csv", low_memory=False)
Modulo1633_REC94_2024 = pd.read_csv("C:/Users/linoc/OneDrive/Encoder/03_partos/01_raws/2024/968-Modulo1633/968-Modulo1633/REC94_2024.csv", low_memory=False)
Modulo1635_RE516171_2024 = pd.read_csv("C:/Users/linoc/OneDrive/Encoder/03_partos/01_raws/2024/968-Modulo1635/968-Modulo1635/RE516171_2024.csv", low_memory=False)
Modulo1640_CSALUD01_2024 = pd.read_csv("C:/Users/linoc/OneDrive/Encoder/03_partos/01_raws/2024/968-Modulo1640/968-Modulo1640/CSALUD01_2024.csv", low_memory=False)

Modulo1640_CSALUD01_2024 = pd.read_csv("C:/Users/linoc/OneDrive/Encoder/03_partos/01_raws/2024/968-Modulo1640/968-Modulo1640/CSALUD01_2024.csv", low_memory=False)
Modulo1640_CSALUD08_2024 = pd.read_csv("C:/Users/linoc/OneDrive/Encoder/03_partos/01_raws/2024/968-Modulo1640/968-Modulo1640/CSALUD08_2024.csv", low_memory=False)


print(f"size file: {get_variable_name(Modulo1631_REC91_2024)}: {Modulo1631_REC91_2024.shape}")
print(f"size file: {get_variable_name(Modulo1632_RE223132_2024)}: {Modulo1632_RE223132_2024.shape}")
print(f"size file: {get_variable_name(Modulo1633_REC41_2024)}: {Modulo1633_REC41_2024.shape}")
print(f"size file: {get_variable_name(Modulo1633_REC94_2024)}: {Modulo1633_REC94_2024.shape}")
print(f"size file: {get_variable_name(Modulo1635_RE516171_2024)}: {Modulo1635_RE516171_2024.shape}")

print(f"size file: {get_variable_name(Modulo1640_CSALUD01_2024)}: {Modulo1640_CSALUD01_2024.shape}")
print(f"size file: {get_variable_name(Modulo1640_CSALUD08_2024)}: {Modulo1640_CSALUD08_2024.shape}")

size file: None: (37117, 343)
size file: None: (34252, 149)
size file: None: (19751, 147)
size file: None: (19751, 61)
size file: None: (34252, 84)
size file: None: (34018, 258)
size file: None: (41309, 53)


## 2.1 match with CASEID level 

In [12]:
Modulo1631_REC91_2024 = Modulo1631_REC91_2024.drop_duplicates()
Modulo1632_RE223132_2024 = Modulo1632_RE223132_2024.drop_duplicates()
Modulo1635_RE516171_2024 = Modulo1635_RE516171_2024.drop_duplicates()

In [13]:
Modulo1631_REC91_2024.shape,Modulo1632_RE223132_2024.shape, Modulo1635_RE516171_2024.shape

((37117, 343), (34252, 149), (34252, 84))

In [14]:
analyze_matches(target_final, Modulo1631_REC91_2024, 'Modulo1631_REC91_2024')
analyze_matches(target_final, Modulo1632_RE223132_2024, 'Modulo1632_RE223132_2024')
analyze_matches(target_final, Modulo1635_RE516171_2024, 'Modulo1635_RE516171_2024')


🔎 Modulo1631_REC91_2024
 - Rows in target_final: 18335
 - Rows in right dataframe (Modulo1631_REC91_2024): 37117
 - Rows after merge: 18335
 ✅ Matches on CASEID: 18335 (100.00%)

🔎 Modulo1632_RE223132_2024
 - Rows in target_final: 18335
 - Rows in right dataframe (Modulo1632_RE223132_2024): 34252
 - Rows after merge: 18335
 ✅ Matches on CASEID: 18335 (100.00%)

🔎 Modulo1635_RE516171_2024
 - Rows in target_final: 18335
 - Rows in right dataframe (Modulo1635_RE516171_2024): 34252
 - Rows after merge: 18335
 ✅ Matches on CASEID: 18335 (100.00%)



In [15]:
print(Modulo1631_REC91_2024.shape[0]), print(Modulo1631_REC91_2024.CASEID.nunique())
print(Modulo1632_RE223132_2024.shape[0]), print(Modulo1632_RE223132_2024.CASEID.nunique())
print(Modulo1635_RE516171_2024.shape[0]), print(Modulo1635_RE516171_2024.CASEID.nunique())

37117
37117
34252
34252
34252
34252


(None, None)

In [16]:
Modulo1631_REC91_2024 = add_prefix_except_caseid(Modulo1631_REC91_2024, 'REC91')
Modulo1632_RE223132_2024 = add_prefix_except_caseid(Modulo1631_REC91_2024, 'RE223132')
Modulo1635_RE516171_2024 = add_prefix_except_caseid(Modulo1635_RE516171_2024, 'RE516171')

# Perform left merges one by one
target_merged = target_final.merge(Modulo1631_REC91_2024, on='CASEID', how='left') \
                            .merge(Modulo1632_RE223132_2024, on='CASEID', how='left') \
                            .merge(Modulo1635_RE516171_2024, on='CASEID', how='left')
target_merged.info(verbose  = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18335 entries, 0 to 18334
Data columns (total 772 columns):
 #    Column                   Non-Null Count  Dtype 
---   ------                   --------------  ----- 
 0    CASEID                   18335 non-null  object
 1    BIDX                     18335 non-null  int64 
 2    Q220A                    18335 non-null  int64 
 3    premature                18335 non-null  int32 
 4    premature_flag           18335 non-null  int32 
 5    REC91_ID1                18335 non-null  int64 
 6    REC91_SVER               18335 non-null  int64 
 7    REC91_SREGION            18335 non-null  int64 
 8    REC91_SSEMES             18335 non-null  int64 
 9    REC91_SPROVIN            18335 non-null  int64 
 10   REC91_SDISTRI            18335 non-null  int64 
 11   REC91_S108N              18335 non-null  object
 12   REC91_S108Y              18335 non-null  object
 13   REC91_S108G              18335 non-null  object
 14   REC91_S111          

## 2.2 match with CASEID and BIRD

In [17]:
target = target.drop_duplicates()
target.shape

(18335, 4)

In [18]:
Modulo1633_REC41_2024.shape, Modulo1633_REC94_2024.shape

((19751, 147), (19751, 61))

In [19]:
#Modulo1632_REC21_2024 = Modulo1632_REC21_2024.drop_duplicates()
Modulo1633_REC41_2024 = Modulo1633_REC41_2024.drop_duplicates()
Modulo1633_REC94_2024  = Modulo1633_REC94_2024.drop_duplicates()


In [20]:
Modulo1633_REC41_2024.shape, Modulo1633_REC94_2024.shape

((19751, 147), (19751, 61))

In [21]:
#analyze_matches_explicit_keys(target, Modulo1632_REC21_2024, 'Modulo1632_REC21_2024', 'BIDX')
analyze_matches_explicit_keys(target, Modulo1633_REC41_2024, 'Modulo1633_REC41_2024', 'MIDX')
analyze_matches_explicit_keys(target, Modulo1633_REC94_2024, 'Modulo1633_REC94_2024', 'IDX94')


🔎 Modulo1633_REC41_2024
 - Rows in target_final: 18335
 - Rows in right dataframe: 19751
 - Matches on CASEID + MIDX: 16983 (92.63%)

🔎 Modulo1633_REC94_2024
 - Rows in target_final: 18335
 - Rows in right dataframe: 19751
 - Matches on CASEID + IDX94: 16983 (92.63%)



### 2.2.1 Modulo1633_REC41_2024

In [22]:
Modulo1633_REC41_2024_M = convert_objects_to_int64_safe(Modulo1633_REC41_2024)
Modulo1633_REC41_2024_M.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19751 entries, 0 to 19750
Data columns (total 147 columns):
 #    Column  Non-Null Count  Dtype 
---   ------  --------------  ----- 
 0    ID1     19751 non-null  int64 
 1    CASEID  19751 non-null  object
 2    MIDX    19751 non-null  int64 
 3    M1      17608 non-null  Int64 
 4    M1A     8385 non-null   Int64 
 5    M1B     6079 non-null   Int64 
 6    M1C     6079 non-null   Int64 
 7    M1D     534 non-null    Int64 
 8    M1E     6079 non-null   Int64 
 9    M2A     17608 non-null  Int64 
 10   M2B     17608 non-null  Int64 
 11   M2C     17608 non-null  Int64 
 12   M2D     17608 non-null  Int64 
 13   M2E     17608 non-null  Int64 
 14   M2F     0 non-null      Int64 
 15   M2G     17608 non-null  Int64 
 16   M2H     0 non-null      Int64 
 17   M2I     0 non-null      Int64 
 18   M2J     0 non-null      Int64 
 19   M2K     17608 non-null  Int64 
 20   M2L     0 non-null      Int64 
 21   M2M     0 non-null      Int64 
 2

In [23]:
Modulo1633_REC41_2024_M = target_final.merge(Modulo1633_REC41_2024_M, 
                                               how='left', left_on=['CASEID', 'BIDX'],
                                                 right_on=['CASEID','MIDX'])

Modulo1633_REC41_2024_M = Modulo1633_REC41_2024_M.drop(['Q220A', 'premature'], axis = 1)

Modulo1633_REC41_2024_M.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18335 entries, 0 to 18334
Data columns (total 149 columns):
 #    Column          Non-Null Count  Dtype  
---   ------          --------------  -----  
 0    CASEID          18335 non-null  object 
 1    BIDX            18335 non-null  int64  
 2    premature_flag  18335 non-null  int32  
 3    ID1             16983 non-null  float64
 4    MIDX            16983 non-null  float64
 5    M1              14982 non-null  Int64  
 6    M1A             6980 non-null   Int64  
 7    M1B             4977 non-null   Int64  
 8    M1C             4977 non-null   Int64  
 9    M1D             481 non-null    Int64  
 10   M1E             4977 non-null   Int64  
 11   M2A             14982 non-null  Int64  
 12   M2B             14982 non-null  Int64  
 13   M2C             14982 non-null  Int64  
 14   M2D             14982 non-null  Int64  
 15   M2E             14982 non-null  Int64  
 16   M2F             0 non-null      Int64  
 17   M2G       

In [24]:
dummy_variables  = get_dummy_variables(Modulo1633_REC41_2024_M)
Modulo1633_REC41_2024_M[dummy_variables].describe().T

,count,mean,std,min,25%,50%,75%,max
premature_flag,18335.0,0.192746,0.394466,0.0,0.0,0.0,0.0,1.0
M2A,14982.0,0.260045,0.438674,0.0,0.0,0.0,1.0,1.0
M2B,14982.0,0.073421,0.260836,0.0,0.0,0.0,0.0,1.0
M2C,14982.0,0.892538,0.30971,0.0,1.0,1.0,1.0,1.0
M2D,14982.0,0.017421,0.130838,0.0,0.0,0.0,0.0,1.0
M2E,14982.0,0.000067,0.00817,0.0,0.0,0.0,0.0,1.0
M2G,14982.0,0.000133,0.011554,0.0,0.0,0.0,0.0,1.0
M2K,14982.0,0.000467,0.021611,0.0,0.0,0.0,0.0,1.0
M2N,14982.0,0.008277,0.090602,0.0,0.0,0.0,0.0,1.0
M3A,16983.0,0.672555,0.469295,0.0,0.0,1.0,1.0,1.0


In [25]:
no_dummy = drop_null_and_list(Modulo1633_REC41_2024_M, dummy_variables)
no_dummy_list =no_dummy.columns.to_list()
no_dummy_list

remove_items = {"M1C", "M1E", "M4","M46","M34", "M19","M6","M7","M8","M9","M11","M13","M14"}
no_dummy_list = [col for col in no_dummy_list if col not in remove_items]


In [26]:
#df_dummy_variables = aggregate_sum_by_caseid(Modulo1633_REC41_2024_M[dummy_variables + ['CASEID']])

df_dummy_variables = Modulo1633_REC41_2024_M[dummy_variables + ['CASEID','BIDX']]
df_dummy_variables.info(verbose = True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18335 entries, 0 to 18334
Data columns (total 36 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   premature_flag  18335 non-null  int32  
 1   M2A             14982 non-null  Int64  
 2   M2B             14982 non-null  Int64  
 3   M2C             14982 non-null  Int64  
 4   M2D             14982 non-null  Int64  
 5   M2E             14982 non-null  Int64  
 6   M2G             14982 non-null  Int64  
 7   M2K             14982 non-null  Int64  
 8   M2N             14982 non-null  Int64  
 9   M3A             16983 non-null  float64
 10  M3B             16983 non-null  float64
 11  M3C             16983 non-null  float64
 12  M3D             16983 non-null  float64
 13  M3E             16983 non-null  float64
 14  M3G             16983 non-null  float64
 15  M3H             16983 non-null  float64
 16  M3K             16983 non-null  float64
 17  M3N             16983 non-null 

In [27]:
Modulo1633_REC41_2024_M [no_dummy_list].describe().T

# categoricas, M10, M15,M42A,M42C, M42D, M42E, M43, M44,M45, M47, M48, M60, M69

,count,mean,std,min,25%,50%,75%,max
BIDX,18335.0,1.151786,0.380799,1.0,1.0,1.0,1.0,4.0
ID1,16983.0,2024.0,0.0,2024.0,2024.0,2024.0,2024.0,2024.0
MIDX,16983.0,1.123182,0.345087,1.0,1.0,1.0,1.0,4.0
M1,14982.0,1.529836,1.279564,0.0,1.0,2.0,2.0,8.0
M1A,6980.0,2.048424,2.21775,0.0,1.0,2.0,2.0,8.0
M1B,4977.0,61.164758,44.97024,1.0,7.0,98.0,98.0,98.0
M1D,481.0,54.590437,44.645649,1.0,8.0,98.0,98.0,98.0
M5,16983.0,17.994701,11.892185,0.0,11.0,17.0,24.0,98.0
M10,16983.0,1.688041,0.778541,1.0,1.0,1.0,2.0,3.0
M15,16983.0,23.007949,8.370836,11.0,21.0,21.0,24.0,96.0


In [28]:
# df_no_dummy_list = aggregate_with_value_suffix(Modulo1633_REC41_2024_M[no_dummy_list], 
#                                                caseid_col="CASEID",
#                                                exclude_cols=['MIDX'])

# df_no_dummy_list.info(verbose=True, show_counts=True)

In [29]:
continue_var_REC41 = ["M6","M7","M8","M9","M11","M13","M14"]

#df_continue_var_REC41 = aggregate_by_caseid_mean(Modulo1633_REC41_2024_M[continue_var_REC41 + ['CASEID']], caseid_col="CASEID")
df_continue_var_REC41 = Modulo1633_REC41_2024_M[continue_var_REC41 + ['CASEID','BIDX']]
df_continue_var_REC41.head()

,M6,M7,M8,M9,M11,M13,M14,CASEID,BIDX
0,12.0,12.0,8.0,8.0,<NA>,1,6,325503101 2,1
1,4.0,4.0,4.0,4.0,<NA>,4,8,325504701 2,1
2,3.0,3.0,3.0,3.0,<NA>,1,11,325505001 1,1
3,6.0,6.0,4.0,4.0,<NA>,1,12,325508901 2,1
4,96.0,18.0,1.0,1.0,<NA>,2,10,325509701 2,1


In [30]:
Modulo1633_REC41_2024_fil = df_dummy_variables.merge(Modulo1633_REC41_2024_M[no_dummy_list], how='left', on='CASEID')
Modulo1633_REC41_2024_fil =Modulo1633_REC41_2024_fil.merge(df_continue_var_REC41, how = 'left', on = 'CASEID')
Modulo1633_REC41_2024_fil.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18335 entries, 0 to 18334
Data columns (total 90 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   premature_flag  18335 non-null  int32  
 1   M2A             14982 non-null  Int64  
 2   M2B             14982 non-null  Int64  
 3   M2C             14982 non-null  Int64  
 4   M2D             14982 non-null  Int64  
 5   M2E             14982 non-null  Int64  
 6   M2G             14982 non-null  Int64  
 7   M2K             14982 non-null  Int64  
 8   M2N             14982 non-null  Int64  
 9   M3A             16983 non-null  float64
 10  M3B             16983 non-null  float64
 11  M3C             16983 non-null  float64
 12  M3D             16983 non-null  float64
 13  M3E             16983 non-null  float64
 14  M3G             16983 non-null  float64
 15  M3H             16983 non-null  float64
 16  M3K             16983 non-null  float64
 17  M3N             16983 non-null 

In [31]:
Modulo1633_REC41_2024_fil = Modulo1633_REC41_2024_fil.drop(columns = ['BIDX_y'], axis=1)

In [32]:
# File name only
output_file = "Modulo1633_REC41_2024_fil_v3.csv"

# Go one level up from the current working directory
base_dir = os.path.dirname(os.getcwd())   # gives "c:\\Users\\linoc\\OneDrive\\Encoder\\03_partos"
output_dir = os.path.join(base_dir, "data\\interim")

#Full path
output_path = os.path.join(output_dir, output_file)

# Save DataFrame
Modulo1633_REC41_2024_fil.to_csv(output_path, index=False, encoding="utf-8-sig")


### 2.2.1 Modulo1633_REC94_2024

In [33]:
Modulo1633_REC94_2024_M = convert_objects_to_int64_safe(Modulo1633_REC94_2024)
Modulo1633_REC94_2024_M.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19751 entries, 0 to 19750
Data columns (total 61 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   ID1       19751 non-null  int64 
 1   CASEID    19751 non-null  object
 2   IDX94     19751 non-null  int64 
 3   S410B     17334 non-null  Int64 
 4   S411B     17426 non-null  Int64 
 5   S411F     17426 non-null  Int64 
 6   S411G     17426 non-null  Int64 
 7   S411H     17426 non-null  Int64 
 8   S411I     17426 non-null  Int64 
 9   S411J     17426 non-null  Int64 
 10  S411K     17426 non-null  Int64 
 11  S411L     17426 non-null  Int64 
 12  S411BA    17054 non-null  Int64 
 13  S411CA    17141 non-null  Int64 
 14  S411DA    15366 non-null  Int64 
 15  S411EA    16355 non-null  Int64 
 16  S413      17608 non-null  Int64 
 17  S422I     16727 non-null  Int64 
 18  S426B     5993 non-null   Int64 
 19  S426E     6957 non-null   Int64 
 20  S426FA    0 non-null      Int64 
 21  S426FB    15

In [34]:
Modulo1633_REC94_2024_M.QI422A_A.value_counts()

QI422A_A
1    16949
2      615
8       44
Name: count, dtype: Int64

In [35]:
Modulo1633_REC94_2024_M = target_final.merge(Modulo1633_REC94_2024_M, 
                                               how='left', left_on=['CASEID', 'BIDX'],
                                                 right_on=['CASEID','IDX94'])

Modulo1633_REC94_2024_M = Modulo1633_REC94_2024_M.drop(['Q220A', 'premature'], axis = 1)

Modulo1633_REC94_2024_M.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18335 entries, 0 to 18334
Data columns (total 63 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CASEID          18335 non-null  object 
 1   BIDX            18335 non-null  int64  
 2   premature_flag  18335 non-null  int32  
 3   ID1             16983 non-null  float64
 4   IDX94           16983 non-null  float64
 5   S410B           14790 non-null  Int64  
 6   S411B           14858 non-null  Int64  
 7   S411F           14858 non-null  Int64  
 8   S411G           14858 non-null  Int64  
 9   S411H           14858 non-null  Int64  
 10  S411I           14858 non-null  Int64  
 11  S411J           14858 non-null  Int64  
 12  S411K           14858 non-null  Int64  
 13  S411L           14858 non-null  Int64  
 14  S411BA          14547 non-null  Int64  
 15  S411CA          14631 non-null  Int64  
 16  S411DA          13108 non-null  Int64  
 17  S411EA          13938 non-null 

In [36]:
Modulo1633_REC94_2024_M.shape, Modulo1633_REC94_2024_M.CASEID.nunique()

((18335, 63), 18335)

In [37]:
dummy_variables_REC94  = get_dummy_variables(Modulo1633_REC94_2024_M)
Modulo1633_REC94_2024_M[dummy_variables_REC94].describe().T

,count,mean,std,min,25%,50%,75%,max
premature_flag,18335.0,0.192746,0.394466,0.0,0.0,0.0,0.0,1.0
S413,14982.0,0.803297,0.397519,0.0,1.0,1.0,1.0,1.0
S426E,6081.0,0.408979,0.491686,0.0,0.0,0.0,1.0,1.0
S426GA,14982.0,0.107796,0.310133,0.0,0.0,0.0,0.0,1.0
S426GB,14982.0,0.089507,0.285484,0.0,0.0,0.0,0.0,1.0
S426GC,14982.0,0.018422,0.134477,0.0,0.0,0.0,0.0,1.0
S426GD,14982.0,0.009278,0.095877,0.0,0.0,0.0,0.0,1.0
S426GE,14982.0,0.050127,0.218214,0.0,0.0,0.0,0.0,1.0
S430D,15095.0,0.998874,0.033541,0.0,1.0,1.0,1.0,1.0
S427DA,14982.0,0.045188,0.207722,0.0,0.0,0.0,0.0,1.0


In [38]:
#df_dummy_variables_REC94 = aggregate_sum_by_caseid(Modulo1633_REC94_2024_M[dummy_variables_REC94 + ['CASEID']])
df_dummy_variables_REC94 =Modulo1633_REC94_2024_M[dummy_variables_REC94 + ['CASEID', 'IDX94']]
df_dummy_variables_REC94.describe().T

,count,mean,std,min,25%,50%,75%,max
premature_flag,18335.0,0.192746,0.394466,0.0,0.0,0.0,0.0,1.0
S413,14982.0,0.803297,0.397519,0.0,1.0,1.0,1.0,1.0
S426E,6081.0,0.408979,0.491686,0.0,0.0,0.0,1.0,1.0
S426GA,14982.0,0.107796,0.310133,0.0,0.0,0.0,0.0,1.0
S426GB,14982.0,0.089507,0.285484,0.0,0.0,0.0,0.0,1.0
S426GC,14982.0,0.018422,0.134477,0.0,0.0,0.0,0.0,1.0
S426GD,14982.0,0.009278,0.095877,0.0,0.0,0.0,0.0,1.0
S426GE,14982.0,0.050127,0.218214,0.0,0.0,0.0,0.0,1.0
S430D,15095.0,0.998874,0.033541,0.0,1.0,1.0,1.0,1.0
S427DA,14982.0,0.045188,0.207722,0.0,0.0,0.0,0.0,1.0


In [39]:
dummy_variables_REC94

['premature_flag',
 'S413',
 'S426E',
 'S426GA',
 'S426GB',
 'S426GC',
 'S426GD',
 'S426GE',
 'S430D',
 'S427DA',
 'S427DB',
 'S427DC',
 'S427DD',
 'S427DE',
 'S427DF',
 'S427DG',
 'S427F',
 'S436C',
 'S441']

In [40]:
no_dummy_REC94 = drop_null_and_list(Modulo1633_REC94_2024_M, dummy_variables_REC94)
no_dummy_list_REC94 =no_dummy_REC94.columns.to_list()
remove_items = {"S411BA","S411CA","S411DA","S411EA","S422I"}
no_dummy_list = [col for col in no_dummy_list_REC94 if col not in remove_items]

In [41]:
no_dummy_list

['CASEID',
 'BIDX',
 'ID1',
 'IDX94',
 'S410B',
 'S411B',
 'S411F',
 'S411G',
 'S411H',
 'S411I',
 'S411J',
 'S411K',
 'S411L',
 'S426B',
 'S426FB',
 'S430C',
 'S431A',
 'S432',
 'S435',
 'S440',
 'S442',
 'S447',
 'QI411_M',
 'QI411F',
 'QI422A_A',
 'QI422A_B',
 'QI422A_C',
 'QI422A_D',
 'QI440B']

In [42]:
# df_no_dummy_list_REC94 = aggregate_with_value_suffix(Modulo1633_REC94_2024_M[no_dummy_list_REC94], caseid_col="CASEID")
# df_no_dummy_list_REC94.describe().T

In [43]:
continue_var = list(remove_items)
#df_continue_var = aggregate_by_caseid_mean(Modulo1633_REC94_2024_M[continue_var + ['CASEID']], caseid_col="CASEID")
df_continue_var = Modulo1633_REC94_2024_M[continue_var + ['CASEID', 'IDX94']]

df_continue_var.head()


,S411BA,S411DA,S411CA,S411EA,S422I,CASEID,IDX94
0,2,2,2,2,0,325503101 2,1.0
1,4,4,4,4,0,325504701 2,1.0
2,2,2,2,2,0,325505001 1,1.0
3,1,1,1,1,0,325508901 2,1.0
4,3,3,3,3,0,325509701 2,1.0


In [44]:
no_dummy_list_REC94

['CASEID',
 'BIDX',
 'ID1',
 'IDX94',
 'S410B',
 'S411B',
 'S411F',
 'S411G',
 'S411H',
 'S411I',
 'S411J',
 'S411K',
 'S411L',
 'S411BA',
 'S411CA',
 'S411DA',
 'S411EA',
 'S422I',
 'S426B',
 'S426FB',
 'S430C',
 'S431A',
 'S432',
 'S435',
 'S440',
 'S442',
 'S447',
 'QI411_M',
 'QI411F',
 'QI422A_A',
 'QI422A_B',
 'QI422A_C',
 'QI422A_D',
 'QI440B']

In [45]:
Modulo1633_REC94_2024_fil = df_dummy_variables_REC94.merge(Modulo1633_REC94_2024_M[no_dummy_list], how='left', on='CASEID')
Modulo1633_REC94_2024_fil= Modulo1633_REC94_2024_fil.merge(df_continue_var, how ='left', on ='CASEID')
Modulo1633_REC94_2024_fil.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18335 entries, 0 to 18334
Data columns (total 55 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   premature_flag  18335 non-null  int32  
 1   S413            14982 non-null  Int64  
 2   S426E           6081 non-null   Int64  
 3   S426GA          14982 non-null  Int64  
 4   S426GB          14982 non-null  Int64  
 5   S426GC          14982 non-null  Int64  
 6   S426GD          14982 non-null  Int64  
 7   S426GE          14982 non-null  Int64  
 8   S430D           15095 non-null  Int64  
 9   S427DA          14982 non-null  Int64  
 10  S427DB          14982 non-null  Int64  
 11  S427DC          14982 non-null  Int64  
 12  S427DD          14982 non-null  Int64  
 13  S427DE          14982 non-null  Int64  
 14  S427DF          14982 non-null  Int64  
 15  S427DG          14982 non-null  Int64  
 16  S427F           4837 non-null   Int64  
 17  S436C           16835 non-null 

In [46]:
Modulo1633_REC94_2024_fil.drop(columns = ['IDX94_y'], axis=1, inplace=True)

In [47]:
# File name only
output_file = "Modulo1633_REC94_2024_fil_v3.csv"

# Go one level up from the current working directory
base_dir = os.path.dirname(os.getcwd())   # gives "c:\\Users\\linoc\\OneDrive\\Encoder\\03_partos"
output_dir = os.path.join(base_dir, "data\\interim")   

#Full path
output_path = os.path.join(output_dir, output_file)

# Save DataFrame
Modulo1633_REC94_2024_fil.to_csv(output_path, index=False, encoding="utf-8-sig")